# Operational Deep-Dive: Inventory & Supply Chain
**Datathon 2026 - Root Cause Analysis**

This notebook investigates the relationship between stock availability and the 2018-2019 volume collapse. We focus on `inventory.csv` metrics like `stockout_days` and `fill_rate`.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Project configuration
PROCESSED_DIR = Path("../data/processed")
PLOT_DIR = Path("../data/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
pd.options.display.max_columns = None

## 1. Data Loading
We use the optimized Parquet datasets.

In [2]:
print("Loading inventory data...")
inventory = pd.read_parquet(PROCESSED_DIR / "inventory.parquet")

# Ensure date format
inventory['snapshot_date'] = pd.to_datetime(inventory['snapshot_date'])
inventory['year_month'] = inventory['snapshot_date'].dt.to_period('M').astype(str)

print(f"Inventory records: {len(inventory)}")

Loading inventory data...
Inventory records: 60247


## 2. Global Inventory Health (2012-2022)
### Diagnostic Analysis
Did stockouts spike during the volume collapse?

In [3]:
inventory_monthly = inventory.groupby('year_month').agg(
    avg_fill_rate=('fill_rate', 'mean'),
    avg_stockout_days=('stockout_days', 'mean'),
    total_stockout_flags=('stockout_flag', 'sum')
).reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=inventory_monthly['year_month'], y=inventory_monthly['avg_fill_rate'],
                        mode='lines', name='Avg Fill Rate', line=dict(color='green')))

fig.add_trace(go.Scatter(x=inventory_monthly['year_month'], y=inventory_monthly['avg_stockout_days'],
                        mode='lines', name='Avg Stockout Days', line=dict(color='red'),
                        yaxis='y2'))

fig.update_layout(
    title='Supply Chain Performance Trend (2012-2022)',
    xaxis=dict(title='Month'),
    yaxis=dict(title='Fill Rate (%)', range=[0, 1.1]),
    yaxis2=dict(title='Stockout Days', overlaying='y', side='right'),
    legend=dict(x=0.01, y=0.99)
)

fig.add_vrect(x0="2018-06", x1="2019-01", 
              annotation_text="2019 Collapse", 
              fillcolor="yellow", opacity=0.1, line_width=0)

fig.write_image(PLOT_DIR / "09_inventory_performance.png")
fig.show()

**Diagnostic Insight:** 
If `Avg Fill Rate` drops or `Stockout Days` spike during the yellow-shaded region, the 'Stockout Hypothesis' is confirmed. This means the revenue loss was due to **availability**, not lack of demand.

## 3. Stockout Risk by Category
### Prescriptive Analysis
Which categories are most prone to stockouts?

In [4]:
cat_inventory = inventory.groupby('category').agg(
    stockout_potential=('stockout_flag', 'mean'),
    avg_days_supply=('days_of_supply', 'mean')
).reset_index().sort_values('stockout_potential', ascending=False)

fig = px.bar(cat_inventory, x='category', y='stockout_potential', 
             title='Stockout Probability by Category',
             color='avg_days_supply', 
             labels={'stockout_potential': 'Probability', 'avg_days_supply': 'Avg Days Supply'})
fig.write_image(PLOT_DIR / "10_category_stockout_risk.png")
fig.show()

**Prescriptive Insight:** 
High stockout probability with low 'Days Supply' indicates a need for more aggressive reordering for those categories. High stockout with high days supply indicates a **supplier reliability issue**.

## 4. Sell-Through-Rate (STR) Efficiency
How fast is inventory turning into sales?

In [5]:
fig = px.scatter(inventory.sample(5000), x='stock_on_hand', y='sell_through_rate', 
                 color='category', size='units_received', 
                 hover_name='product_name', 
                 title='Sell-Through-Rate vs Stock Levels (Sampled)')
fig.write_image(PLOT_DIR / "11_inventory_efficiency.png")
fig.show()